In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI


# ============================================================
# 1. LOAD API KEY
# ============================================================

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)


# ============================================================
# 2. AI TEACHER INSTRUCTIONS
# ============================================================

TEACHER_INSTRUCTIONS = """
You are an expert AI Teacher.

Your job is to teach students, not just provide answers.

Teaching rules:
Give only simple explanation of the concept with layman terms.
Answer should be precise 3 lines.
Response format:
Explain the concept simply.
"""


# ============================================================
# 3. CONVERSATION MEMORY
# ============================================================

conversation = [
    {
        "role": "developer",
        "content": TEACHER_INSTRUCTIONS
    }
]


# ============================================================
# 4. AI TEACHER FUNCTION
# ============================================================

def ask_teacher(
    question,
    subject="AI & Data Science",
    student_level="Beginner",
    language="English"
):

    user_message = f"""
Subject: {subject}

Student Level: {student_level}

Preferred Language: {language}

Student Question:
{question}
"""

    conversation.append({
        "role": "user",
        "content": user_message
    })

    response = client.responses.create(
        model="gpt-5.6-luna",
        input=conversation
    )

    answer = response.output_text

    conversation.append({
        "role": "assistant",
        "content": answer
    })

    return answer


# ============================================================
# 5. CHAT APPLICATION
# ============================================================

def start_teacher():

    print("=" * 60)
    print("              AI TEACHER")
    print("=" * 60)

    print("Type 'exit' to stop.\n")

    while True:

        question = input("Student: ")

        if question.lower() == "exit":

            print("\nTeacher: Keep learning! 👋")
            break

        answer = ask_teacher(
            question=question,
            subject="AI & Data Science",
            student_level="Beginner",
            language="English"
        )

        print("\nTeacher:")
        print(answer)

        print("\n" + "-" * 60)


# ============================================================
# 6. RUN
# ============================================================

if __name__ == "__main__":
    start_teacher()

              AI TEACHER
Type 'exit' to stop.



Student:  what is AI?



Teacher:
AI (Artificial Intelligence) is the ability of computers to think, learn, and make decisions like humans.  
It uses data and rules to solve problems, recognize patterns, and perform tasks.  
Examples include voice assistants, recommendation systems, and self-driving cars.

------------------------------------------------------------


Student:  tell about Machine leanering



Teacher:
Machine learning is a part of AI where computers learn from data and improve without being directly programmed for every task.  
It finds patterns in examples to make predictions or decisions.  
Examples include spam detection, movie recommendations, and face recognition.

------------------------------------------------------------


Student:  exit



Teacher: Keep learning! 👋


# Stage2 -  input Quries- Generate

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI


# ============================================================
# 1. LOAD ENVIRONMENT
# ============================================================

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)


# ============================================================
# 2. FILE CONFIGURATION
# ============================================================

INPUT_CSV = "ai_teacher_20_questions.csv"
OUTPUT_CSV = "ai_teacher_results.csv"


# ============================================================
# 3. LOAD DATASET
# ============================================================

df = pd.read_csv(INPUT_CSV)

print("Dataset loaded")
print(f"Number of questions: {len(df)}")


# ============================================================
# 4. AI TEACHER INSTRUCTIONS
# ============================================================

TEACHER_INSTRUCTIONS = """
You are an expert AI Teacher.

Your job is to teach students, not just provide answers.

Teaching rules:
Give only simple explanation of the concept with layman terms.
Answer should be precise 3 lines.
Response format:
Explain the concept simply.
"""


# ============================================================
# 5. GENERATE ANSWER
# ============================================================

def generate_answer(question):

    response = client.responses.create(
        model="gpt-5.6-luna",
        instructions=TEACHER_INSTRUCTIONS,
        input=question
    )

    return response.output_text.strip()


# ============================================================
# 6. GENERATE ANSWERS FOR ALL QUESTIONS
# ============================================================

generated_answers = []

for index, row in df.iterrows():

    question = row["User Query"]

    print(f"Processing {index + 1}/{len(df)}")
    print(f"Question: {question}")

    try:

        answer = generate_answer(question)

    except Exception as e:

        answer = f"ERROR: {str(e)}"

    generated_answers.append(answer)


# ============================================================
# 7. APPEND GENERATED ANSWER TO DATAFRAME
# ============================================================

df["GeneratedAnswer"] = generated_answers


# ============================================================
# 8. SELECT FINAL COLUMNS
# ============================================================

df = df[
    [
        "User Query",
        "Teacher Response",
        "GeneratedAnswer"
    ]
]


# ============================================================
# 9. SAVE RESULT
# ============================================================

df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 10. DISPLAY RESULT
# ============================================================

print("\n" + "=" * 70)
print("AI TEACHER TEST COMPLETED")
print("=" * 70)

print(df.to_string(index=False))

print("\nResult saved to:")
print(OUTPUT_CSV)

Dataset loaded
Number of questions: 20
Processing 1/20
Question: What is AI?
Processing 2/20
Question: How does AI differ from traditional programming?
Processing 3/20
Question: What is Machine Learning?
Processing 4/20
Question: What is Deep Learning?
Processing 5/20
Question: What is the difference between AI, ML, and Deep Learning?
Processing 6/20
Question: What is training data?
Processing 7/20
Question: What is a Machine Learning model?
Processing 8/20
Question: What is supervised learning?
Processing 9/20
Question: What is unsupervised learning?
Processing 10/20
Question: What is reinforcement learning?
Processing 11/20
Question: What is an LLM?
Processing 12/20
Question: What is Generative AI?
Processing 13/20
Question: What is a prompt?
Processing 14/20
Question: What is Prompt Engineering?
Processing 15/20
Question: What is RAG?
Processing 16/20
Question: Why is RAG useful?
Processing 17/20
Question: What is a vector database?
Processing 18/20
Question: What are embeddings?
Pr

# Stage3 - LLM Evaluation- Answer Quality

In [3]:
import pandas as pd

from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.evaluate import AsyncConfig

# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv("ai_teacher_results.csv")


# ============================================================
# 2. CREATE TEST CASES
# ============================================================

test_cases = [
    LLMTestCase(
        input=row["User Query"],
        actual_output=row["GeneratedAnswer"],
        expected_output=row["Teacher Response"]
    )
    for _, row in df.iterrows()
]


# ============================================================
# 3. DEFINE METRICS
# ============================================================

correctness = GEval(
    name="Answer Correctness",

    model="gpt-4.1",

    criteria="""
    Evaluate whether the generated answer is factually correct
    compared with the expected teacher answer.

    Give a high score when the important meaning and facts are correct.

    Penalize:
    - incorrect facts
    - contradictory information
    - missing important information

    Minor wording differences should not reduce the score.
    """,

    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT
    ],

    async_mode=AsyncConfig(run_async=False)
)


relevancy = AnswerRelevancyMetric(
    model="gpt-4.1",

    include_reason=False,

    async_mode=False
)


helpfulness = GEval(
    name="Helpfulness",

    model="gpt-4.1",

    criteria="""
    Evaluate how helpful the generated answer is for a beginner student.

    A helpful answer should:

    1. Directly answer the student's question.
    2. Be easy to understand.
    3. Provide enough explanation.
    4. Avoid unnecessary information.
    5. Help the student understand the concept.
    """,

    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT
    ],

    async_mode=AsyncConfig(run_async=False)
)


# ============================================================
# 4. RUN DEEPEVAL
# ============================================================

result = evaluate(
    test_cases=test_cases,

    metrics=[
        correctness,
        relevancy,
        helpfulness
    ],

    # IMPORTANT FOR JUPYTER / ANACONDA
    async_config=AsyncConfig(run_async=False)
)



✨ You're running DeepEval's latest Answer Correctness [GEval] Metric! (using gpt-4.1, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1, strict=False, async_mode=False)...

✨ You're running DeepEval's latest Helpfulness [GEval] Metric! (using gpt-4.1, strict=False, async_mode=False)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_4 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_5 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_6 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_7 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_8 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_9 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_10 (Passed 3 metrics)                                

⚠ WARNING: No hyperparameters logged.
» ]8;id=707014;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 251.72s | token cost: 0.127002 USD)
» Test Results (20 total tests):
   » Pass Rate: 100.0% | Passed: 20 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [4]:
dir(result)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_extra_info__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_on_complete__',
 '__pydantic_parent_namespace__',
 '__pydantic_post_init__',
 '__pydantic_private__',
 '__pydantic_root_model__

In [6]:
result.test_results

[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Correctness [GEval]', threshold=0.5, success=True, score=0.8901920667961543, reason='The Actual Output accurately conveys the main facts from the Expected Output, including that AI enables computers to perform tasks requiring human intelligence, such as learning, reasoning, and decision-making. It also adds relevant examples and details like image recognition and language understanding, which align with the intent of the Expected Output. There are no incorrect facts or contradictions, but the Actual Output is slightly more detailed than required, which does not detract from accuracy.', strict_mode=False, flaky=False, evaluation_model='gpt-4.1', error=None, evaluation_cost=0.0026379999999999997, input_tokens=595, output_tokens=181, verbose_logs='Criteria:\n\n    Evaluate whether the generated answer is factually correct\n    compared with the expected teacher answer.\n\n    Give a high score when the imp

In [7]:

for i, test_result in enumerate(result.test_results):

    metrics_data = test_result.metrics_data

    df.loc[i, "AnswerCorrectness"] = metrics_data[0].score
    df.loc[i, "AnswerRelevancy"] = metrics_data[1].score
    df.loc[i, "Helpfulness"] = metrics_data[2].score


# ============================================================
# 3. CALCULATE OVERALL ANSWER QUALITY
# ============================================================

df["AnswerQuality"] = (
    df["AnswerCorrectness"]
    + df["AnswerRelevancy"]
    + df["Helpfulness"]
) / 3


# ============================================================
# 4. ROUND SCORES
# ============================================================

metric_columns = [
    "AnswerCorrectness",
    "AnswerRelevancy",
    "Helpfulness",
    "AnswerQuality"
]

df[metric_columns] = df[metric_columns].round(3)


# ============================================================
# 5. AVERAGES
# ============================================================

average_correctness = df["AnswerCorrectness"].mean()
average_relevancy = df["AnswerRelevancy"].mean()
average_helpfulness = df["Helpfulness"].mean()
average_quality = df["AnswerQuality"].mean()


# ============================================================
# 6. QUESTION-WISE RESULTS
# ============================================================

print("\n")
print("=" * 120)
print("                         AI TEACHER ANSWER QUALITY")
print("=" * 120)

display_df = df[
    [
        "User Query",
        "AnswerCorrectness",
        "AnswerRelevancy",
        "Helpfulness",
        "AnswerQuality"
    ]
]


# ============================================================
# 8. SAVE FINAL RESULT
# ============================================================

df.to_csv(
    "ai_teacher_evaluation.csv",
    index=False,
    encoding="utf-8-sig"
)





                         AI TEACHER ANSWER QUALITY


In [8]:
df

,User Query,Teacher Response,GeneratedAnswer,AnswerCorrectness,AnswerRelevancy,Helpfulness,AnswerQuality
0,What is AI?,AI (Artificial Intelligence) is the ability of...,AI (Artificial Intelligence) is technology tha...,0.890,1.000,1.000,0.963
1,How does AI differ from traditional programming?,Traditional programs follow fixed rules writte...,Traditional programming follows explicit rules...,1.000,1.000,1.000,1.000
2,What is Machine Learning?,Machine Learning is a branch of AI where compu...,Machine Learning is a way for computers to lea...,0.900,1.000,1.000,0.967
3,What is Deep Learning?,Deep Learning is a type of Machine Learning th...,Deep Learning is a type of AI that uses artifi...,0.729,1.000,1.000,0.910
4,"What is the difference between AI, ML, and Dee...",AI is the broad field of making machines intel...,AI is the broad idea of making computers perfo...,0.900,1.000,1.000,0.967
5,What is training data?,Training data is the data used to teach a Mach...,Training data is the information used to teach...,0.900,1.000,1.000,0.967
6,What is a Machine Learning model?,A Machine Learning model is a mathematical sys...,A machine learning model is a computer program...,0.900,1.000,1.000,0.967
7,What is supervised learning?,Supervised learning trains a model using examp...,Supervised learning is a type of machine learn...,0.900,1.000,0.998,0.966
8,What is unsupervised learning?,Unsupervised learning works with data that doe...,Unsupervised learning is a type of machine lea...,0.969,1.000,1.000,0.990
9,What is reinforcement learning?,Reinforcement learning teaches an agent throug...,Reinforcement learning is a type of machine le...,1.000,1.000,0.900,0.967


In [15]:
average_result=pd.DataFrame(columns=["average_correctness","average_relevancy","average_helpfulness","average_quality"],
                            data=[[average_correctness,average_relevancy,average_helpfulness,average_quality]])

In [16]:
average_result

,average_correctness,average_relevancy,average_helpfulness,average_quality
0,0.87875,1.0,0.9985,0.9591
